# Sample Checklist Extraction Review

This notebook will inspect the loader and truck checklist files and compare the extracted fields against the expected checklist structure.

## 1. Load Sample Checklist Documents
Read the sample DOCX files and extract the raw text contents for inspection.

In [ ]:
import zipfile
import os
from xml.etree import ElementTree as ET

sample_files = [
    "sample/FM-EN-180- LOADER.docx",
    "sample/FM-EN-181- TRUCKS.docx",
]

for sample_file in sample_files:
    path = os.path.normpath(sample_file)
    print(f"---\nFILE: {path}")
    if not os.path.exists(path):
        print("MISSING FILE")
        continue
    with zipfile.ZipFile(path, "r") as z:
        if "word/document.xml" not in z.namelist():
            print("word/document.xml not found")
            continue
        xml = z.read("word/document.xml")
        root = ET.fromstring(xml)
        ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
        texts = [t.text for t in root.findall('.//w:t', ns) if t.text]
        raw_text = " ".join(texts)
        print(raw_text[:5000])
        print("...\n")

## 2. Parse Loader Checklist Fields
Extract the expected checklist fields from the loader document text and identify the form sections.

In [ ]:
from collections import OrderedDict
import re

loader_path = os.path.normpath("sample/FM-EN-180- LOADER.docx")

with zipfile.ZipFile(loader_path, "r") as z:
    xml = z.read("word/document.xml")
    root = ET.fromstring(xml)
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    texts = [t.text for t in root.findall('.//w:t', ns) if t.text]
    loader_text = " ".join(texts)

expected_fields = [
    "machine number",
    "operator name",
    "mine number",
    "permit number",
    "date",
    "shift",
    "start engine hours",
    "end engine hours",
    "start transmission hours",
    "end transmission hours",
    "engineering/maintenance daily checks",
    "activity code",
    "from time",
    "to time",
    "workplace",
    "ore/waste",
    "loads",
    "remarks",
]

field_matches = OrderedDict()
for field in expected_fields:
    regex = re.compile(re.escape(field), re.IGNORECASE)
    field_matches[field] = bool(regex.search(loader_text))

print("LOADER FIELD PRESENCE")
for field, present in field_matches.items():
    print(f"{field}: {'FOUND' if present else 'MISSING'}")

print("\nLOADER TEXT PREVIEW:\n")
print(loader_text[:2000])

## 3. Parse Truck Checklist Fields
Extract the expected checklist fields from the truck document text and identify the form sections.

In [ ]:
truck_path = os.path.normpath("sample/FM-EN-181- TRUCKS.docx")

with zipfile.ZipFile(truck_path, "r") as z:
    xml = z.read("word/document.xml")
    root = ET.fromstring(xml)
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    texts = [t.text for t in root.findall('.//w:t', ns) if t.text]
    truck_text = " ".join(texts)

field_matches = OrderedDict()
for field in expected_fields:
    regex = re.compile(re.escape(field), re.IGNORECASE)
    field_matches[field] = bool(regex.search(truck_text))

print("TRUCK FIELD PRESENCE")
for field, present in field_matches.items():
    print(f"{field}: {'FOUND' if present else 'MISSING'}")

print("\nTRUCK TEXT PREVIEW:\n")
print(truck_text[:2000])

## 4. Normalize Extracted Checklist Data
Normalize the extracted raw text into a consistent structured schema and detect messy handwritten forms.

In [ ]:
def normalize_field_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

normalized_loader = normalize_field_name(loader_text)
normalized_truck = normalize_field_name(truck_text)

print('Normalized loader text sample:')
print(normalized_loader[:1000])
print('\n---\nNormalized truck text sample:')
print(normalized_truck[:1000])

## 5. Validate and Compare Extraction Results
Compare the extracted checklist data against expected checklist field structures and check for completeness and accuracy.

In [ ]:
def summarize_field_presence(sample_text: str, fields: list[str]) -> dict:
    summary = {}
    for field in fields:
        summary[field] = bool(re.search(re.escape(field), sample_text, re.IGNORECASE))
    return summary

loader_summary = summarize_field_presence(loader_text, expected_fields)
truck_summary = summarize_field_presence(truck_text, expected_fields)

print('Loader Checklist Field Results:')
for field, found in loader_summary.items():
    print(f'{field}: {found}')

print('\nTruck Checklist Field Results:')
for field, found in truck_summary.items():
    print(f'{field}: {found}')

## 6. Export Clean Structured Output
Write the cleaned, validated checklist data to a structured output format such as JSON or CSV.

In [ ]:
import json

output = {
    'loader': {
        'raw_text_sample': loader_text[:5000],
        'fields_found': loader_summary,
    },
    'truck': {
        'raw_text_sample': truck_text[:5000],
        'fields_found': truck_summary,
    },
}
with open('sample_checklist_extraction_result.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print('Structured extraction output written to sample_checklist_extraction_result.json')

In [ ]:
print('notebook execution test')